# Exploratory Data Analysis: Business Entity Resolution

## Executive Summary of Key Findings

1. **Dataset Overview & Row Counts**:
   - **Source 1 (Reference)**: Deduplicated master reference table containing primary entity profiles.
   - **Source 2 & Source 3**: Heterogeneous secondary sources with varying completeness and noisy entity attributes.
   - **Ground Truth**: Maps each `S1-` entity to a list of matching `S2-` and `S3-` entity IDs.

2. **Country Distribution**:
   - Training dataset covers **US** and **India** across all three sources.
   - Note: Per challenge guidelines, the test set additionally contains **France**, requiring open-vocabulary country handling.

3. **Match Cardinality & Singleton Distribution**:
   - **Singletons (0 matches)**: A significant portion of Source 1 entities have no valid matches in S2 or S3. Per the evaluation metric ($F_{0.5}$), correctly predicting empty match sets for singletons is heavily rewarded (1.0 score per correct singleton).
   - **Single Matches (1 match)**: Source 1 entity pairs with exactly one counterpart in either S2 or S3.
   - **Multi-Matches (>1 match)**: Entities match records in both S2 and S3, or multiple entries due to duplicate listings in secondary sources.

4. **Cross-Source Match Rates**:
   - Source 1 entities exhibit roughly balanced average match density across Source 2 and Source 3.

5. **Observed Noise Patterns in Matched Pairs**:
   - **Legal Suffixes**: Frequent divergence between abbreviations and full forms (`Pvt Ltd` vs `Private Limited`, `Corp` vs `Corporation`, `Inc` vs `Incorporated`, `Co` vs `Company`, `&` vs `and`).
   - **Street Abbreviations**: Shortened terms (`Rd` vs `Road`, `St` vs `Street`, `Ave` vs `Avenue`, `Bldg`, `Ste`, `Apt`).
   - **Landmark Clauses**: Heavy injection of relative landmark phrases in Indian addresses (`Near CST Station`, `Opp Vidhan Bhavan`, `Behind Kaikondrahalli Lake`).
   - **PIN / Postal Codes**: Varied placement, truncation, or omission across sources.

6. **Integrity & Anomaly Checks**:
   - Verified 100% strict prefix compliance: `S1-` for Source 1, `S2-` for Source 2, `S3-` for Source 3.
   - Zero ID collisions across source files.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import load_training_data
from src.config import TRAIN_PATH

print(f"Project root: {PROJECT_ROOT}")
data = load_training_data(PROJECT_ROOT / "dataset/train")
s1 = data["train_source1"]
s2 = data["train_source2"]
s3 = data["train_source3"]
gt = data["train_ground_truth"]
print("Data loaded successfully!")

Project root: D:\business_entity_resolution
Data loaded successfully!


In [ ]:
# Row counts for each source file
counts = {
    "train_source1": len(s1),
    "train_source2": len(s2),
    "train_source3": len(s3),
    "train_ground_truth": len(gt)
}
counts_df = pd.DataFrame(list(counts.items()), columns=["Source File", "Row Count"])
print("--- Dataset Row Counts ---")
print(counts_df.to_string(index=False))

--- Dataset Row Counts ---
       Source File  Row Count
     train_source1         20
     train_source2         16
     train_source3         12
train_ground_truth         20


In [ ]:
# Distribution of country values in source1, source2, source3
print("--- Country Distribution Across Sources ---")
dist_s1 = s1["country"].value_counts().rename("Source 1")
dist_s2 = s2["country"].value_counts().rename("Source 2")
dist_s3 = s3["country"].value_counts().rename("Source 3")

country_dist = pd.concat([dist_s1, dist_s2, dist_s3], axis=1).fillna(0).astype(int)
country_dist["Total"] = country_dist.sum(axis=1)
print(country_dist)

--- Country Distribution Across Sources ---
         Source 1  Source 2  Source 3  Total
country                                     
India          10         9         6     25
US             10         7         6     23


In [ ]:
# How many source1 entities have 0 matches, 1 match, >1 match
gt["num_matches"] = gt["matched_entity_ids"].apply(len)

def categorize_matches(n):
    if n == 0:
        return "0 matches (Singletons)"
    elif n == 1:
        return "1 match"
    else:
        return ">1 matches"

gt["match_category"] = gt["num_matches"].apply(categorize_matches)
match_summary = gt["match_category"].value_counts().reindex(["0 matches (Singletons)", "1 match", ">1 matches"]).fillna(0).astype(int)

print("--- Source 1 Match Cardinality Histogram ---")
for cat, count in match_summary.items():
    pct = (count / len(gt)) * 100
    bar = "█" * int(pct // 4)
    print(f"{cat:24} : {count:3d} ({pct:5.1f}%) {bar}")

--- Source 1 Match Cardinality Histogram ---
0 matches (Singletons)   :   3 ( 15.0%) ███
1 match                  :   8 ( 40.0%) ██████████
>1 matches               :   9 ( 45.0%) ███████████


In [ ]:
# Average number of source2 vs source3 matches per source1 entity
gt["s2_matches"] = gt["matched_entity_ids"].apply(lambda ids: sum(1 for x in ids if x.startswith("S2-")))
gt["s3_matches"] = gt["matched_entity_ids"].apply(lambda ids: sum(1 for x in ids if x.startswith("S3-")))

avg_s2 = gt["s2_matches"].mean()
avg_s3 = gt["s3_matches"].mean()
total_avg = gt["num_matches"].mean()

print("--- Average Matches per Source 1 Entity ---")
print(f"Average Source 2 matches per S1 entity: {avg_s2:.3f}")
print(f"Average Source 3 matches per S1 entity: {avg_s3:.3f}")
print(f"Average Total matches per S1 entity:    {total_avg:.3f}")

--- Average Matches per Source 1 Entity ---
Average Source 2 matches per S1 entity: 0.750
Average Source 3 matches per S1 entity: 0.550
Average Total matches per S1 entity:    1.300


In [ ]:
# 10 examples of matched pairs side-by-side for visual inspection of noise
s2_lookup = s2.set_index("entity_id")
s3_lookup = s3.set_index("entity_id")
s1_lookup = s1.set_index("entity_id")

pairs = []
for _, row in gt.iterrows():
    s1_id = row["source1_entity_id"]
    s1_rec = s1_lookup.loc[s1_id]
    for mid in row["matched_entity_ids"]:
        if mid.startswith("S2-") and mid in s2_lookup.index:
            m_rec = s2_lookup.loc[mid]
            m_source = "Source 2"
        elif mid.startswith("S3-") and mid in s3_lookup.index:
            m_rec = s3_lookup.loc[mid]
            m_source = "Source 3"
        else:
            continue
        pairs.append({
            "S1 ID": s1_id,
            "S1 Name": s1_rec["business_name"],
            "S1 Address": s1_rec["business_address"],
            "Matched ID": mid,
            "Matched Source": m_source,
            "Matched Name": m_rec["business_name"],
            "Matched Address": m_rec["business_address"]
        })

pairs_df = pd.DataFrame(pairs)
sample_n = min(10, len(pairs_df))
sample_pairs = pairs_df.sample(n=sample_n, random_state=42).reset_index(drop=True)

print(f"--- {sample_n} Matched Pairs Side-by-Side Comparison ---")
for i, r in sample_pairs.iterrows():
    print(f"Example {i+1} [{r['S1 ID']} <-> {r['Matched ID']} ({r['Matched Source']})]")
    print(f"  S1:      {r['S1 Name']}")
    print(f"           {r['S1 Address']}")
    print(f"  MATCHED: {r['Matched Name']}")
    print(f"           {r['Matched Address']}")
    print("-" * 70)

--- 10 Matched Pairs Side-by-Side Comparison ---
Example 1 [S1-00006 <-> S2-00005 (Source 2)]
  S1:      State Bank of India
           State Bank Bhavan, Madame Cama Road, Nariman Point, Mumbai 400021
  MATCHED: SBI Bank
           Madame Cama Rd, Opp Air India Bldg, Nariman Point, Mumbai 400021
----------------------------------------------------------------------
Example 2 [S1-00012 <-> S2-00010 (Source 2)]
  S1:      Microsoft Corporation
           One Microsoft Way, Redmond, WA 98052
  MATCHED: Microsoft Corp.
           1 Microsoft Way, Bldg 92, Redmond, WA 98052
----------------------------------------------------------------------
Example 3 [S1-00001 <-> S2-00001 (Source 2)]
  S1:      Tata Consultancy Services
           TCS House, Raveline Street, Fort, Mumbai 400001
  MATCHED: Tata Consultancy Services Ltd.
           Raveline St, Near CST Station, Fort, Mumbai 400001
----------------------------------------------------------------------
Example 4 [S1-00018 <-> S3-00011 (So

In [ ]:
# Check for any entity_id overlaps or anomalies
s1_ids = set(s1["entity_id"])
s2_ids = set(s2["entity_id"])
s3_ids = set(s3["entity_id"])

print("--- Entity ID Overlap & Anomaly Checks ---")
# Check 1: Overlaps
overlap_1_2 = s1_ids.intersection(s2_ids)
overlap_1_3 = s1_ids.intersection(s3_ids)
overlap_2_3 = s2_ids.intersection(s3_ids)
print(f"Overlap between S1 and S2: {len(overlap_1_2)} IDs")
print(f"Overlap between S1 and S3: {len(overlap_1_3)} IDs")
print(f"Overlap between S2 and S3: {len(overlap_2_3)} IDs")

# Check 2: Prefix compliance
s1_prefix_violations = [x for x in s1_ids if not x.startswith("S1-")]
s2_prefix_violations = [x for x in s2_ids if not x.startswith("S2-")]
s3_prefix_violations = [x for x in s3_ids if not x.startswith("S3-")]
print(f"S1 prefix violations: {len(s1_prefix_violations)}")
print(f"S2 prefix violations: {len(s2_prefix_violations)}")
print(f"S3 prefix violations: {len(s3_prefix_violations)}")

# Check 3: Ground truth reference validity
all_secondary = s2_ids.union(s3_ids)
invalid_gt_refs = []
for _, row in gt.iterrows():
    for mid in row["matched_entity_ids"]:
        if mid not in all_secondary:
            invalid_gt_refs.append((row["source1_entity_id"], mid))
print(f"Invalid ground truth references (pointing to non-existent IDs): {len(invalid_gt_refs)}")

if not overlap_1_2 and not overlap_1_3 and not overlap_2_3 and not s1_prefix_violations and not invalid_gt_refs:
    print("ALL INTEGRITY CHECKS PASSED: No anomalies found!")

--- Entity ID Overlap & Anomaly Checks ---
Overlap between S1 and S2: 0 IDs
Overlap between S1 and S3: 0 IDs
Overlap between S2 and S3: 0 IDs
S1 prefix violations: 0
S2 prefix violations: 0
S3 prefix violations: 0
Invalid ground truth references (pointing to non-existent IDs): 0
ALL INTEGRITY CHECKS PASSED: No anomalies found!
